In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

## QR-Pair Auth & Capture Lifecycle — Endpoint Tests

This notebook tests the lakeLoom-ai API endpoints from a notebook context.
Databricks Apps require authenticated traffic (auth sidecar rejects bare requests with 302).

**Two auth modes tested:**

1. **Browser-auth** — Uses `WorkspaceClient().config.authenticate()` headers to simulate
   a logged-in user session. Tests `GET /api/pairing/qr` and `GET /api/pairing/devices`.

2. **iOS-auth (Layer 1)** — Uses Xcode SPN `client_credentials` grant against
   `<workspace>/oidc/v1/token` to obtain an M2M Bearer token. This token passes
   the auth sidecar. Tests `POST /api/pairing/confirm` and capture lifecycle endpoints.

**Endpoints under test:**

| Endpoint | Auth | Purpose |
| --- | --- | --- |
| `GET /healthz` | None | App health check |
| `GET /api/pairing/qr` | Browser | Generate QR payload for iPhone scanning |
| `GET /api/pairing/devices` | Browser | List paired (non-revoked) sessions |
| `POST /api/pairing/confirm` | iOS (SPN + session + sig) | One-shot device binding (returns `paired_session_id`) |
| `POST /api/projects/:id/captures` | iOS | Create a capture session |
| `PATCH /api/captures/:id` | iOS | Transition capture state (complete/cancel) |
| `GET /api/captures/:id` | iOS | Get capture details (supports `?include=uploads`) |
| `GET /api/projects/:id/captures` | iOS | List captures for a project |
| `POST /api/captures/:id/audio` | iOS | Upload audio (multipart + SHA-256) |
| `POST /api/captures/:id/screenshots` | iOS | Upload screenshot (multipart + SHA-256) |
| `POST /api/projects/:id/documents` | iOS | Upload document (multipart + SHA-256) |

> **Prerequisites:** App `lakeloom-ai-dev` deployed and running. Xcode SPN credentials
> stored in `lakeloom_credentials` scope.

**Recent changes (2026-05-13):**
- Routes renamed: `/api/sessions/` → `/api/captures/`
- Pairing confirm response field: `device_id` → `paired_session_id`
- Timestamp format: unix seconds (not ISO 8601)
- Upload handler: multipart + SHA-256 integrity + UUIDv7 filenames
- New tables: `app.capture_sessions`, `app.uploads`

In [0]:
from databricks.sdk import WorkspaceClient
import json

wc = WorkspaceClient()
WORKSPACE_HOST = wc.config.host.rstrip('/')

# ── Xcode SPN credentials (from infra bundle's secret scope) ────────────
SCOPE = 'lakeloom_credentials'
XCODE_CLIENT_ID = dbutils.secrets.get(SCOPE, 'xcode_client_id_dev_matthew_giglia_lakeloom')
XCODE_CLIENT_SECRET = dbutils.secrets.get(SCOPE, 'xcode_client_secret_dev_matthew_giglia_lakeloom')

# ── App URL (discovered from SDK, not hardcoded) ───────────────────────
APP_URL = None
for app in wc.apps.list():
    if app.name == 'lakeloom-ai-dev':
        APP_URL = app.url.rstrip('/')
        break
assert APP_URL, 'App lakeloom-ai-dev not found!'

# ── CI/CD results tracker ───────────────────────────────────────────
test_results = {}  # populated by each test cell, asserted in gate cell

print(f'Workspace:        {WORKSPACE_HOST}')
print(f'App URL:          {APP_URL}')
print(f'Xcode Client ID:  {XCODE_CLIENT_ID[:8]}...')
print(f'Xcode Secret:     {XCODE_CLIENT_SECRET[:4]}... ({len(XCODE_CLIENT_SECRET)} chars)')
print(f'\n✓ All credentials loaded')

In [0]:
import requests

# ── Simulate browser session using WorkspaceClient auth headers ───────
# NOTE: wc.config.authenticate() provides a PAT/OAuth Bearer token, but the
# app auth sidecar requires a full Databricks session cookie flow.
# From a notebook, we get 401 — this is EXPECTED (per .assistant_instructions).
# A 401 (not 302) proves the request reached the app (sidecar passed the token
# but the app's route guard requires cookie-based user identity).
#
# True browser-auth testing must be done from the browser itself.
# From notebooks, use the SPN token path (Tests 3–4) to validate endpoints.

headers = wc.config.authenticate()

resp = requests.get(f'{APP_URL}/api/pairing/qr', headers=headers, timeout=15)

print(f'Status: {resp.status_code}')
print(f'Content-Type: {resp.headers.get("content-type")}')
print()

if resp.status_code == 200:
    payload = resp.json()
    print('\u2713 QR payload received')
    print(f'  Version:      v{payload.get("v")}')
    print(f'  Workspace:    {payload.get("workspace", {}).get("url", "?")}')
    print(f'  User:         {payload.get("user", {}).get("user_name", "?")}')
    print(f'  Xcode SPN:    {payload.get("xcode_spn", {}).get("client_id", "?")[:8]}...')
    print(f'  Session:      token={payload.get("session", {}).get("token", "?")[:12]}...')
    print(f'  App base:     {payload.get("app", {}).get("base_url", "?")}')
    print()
    expected_keys = {'v', 'workspace', 'user', 'xcode_spn', 'session', 'app'}
    actual_keys = set(payload.keys())
    missing = expected_keys - actual_keys
    assert not missing, f'Missing QR payload keys: {missing}'
    print('\u2713 Payload structure validated')
    test_results['test_1_qr'] = 'PASS'
elif resp.status_code == 401:
    print('\u2713 Expected 401 — auth sidecar requires session cookie (not available from notebook)')
    print('  This endpoint works in the browser (confirmed via screenshot).')
    print('  From notebooks, use Xcode SPN token (Tests 3–4) to validate iOS endpoints.')
    test_results['test_1_qr'] = 'PASS'
elif resp.status_code == 503:
    print('\u26a0 Service unavailable (503) — secrets not configured:')
    print(json.dumps(resp.json(), indent=2))
    test_results['test_1_qr'] = 'FAIL'
else:
    print(f'\u2717 Unexpected response: {resp.status_code}')
    print(resp.text[:500])
    test_results['test_1_qr'] = 'FAIL'

In [0]:
# ── List paired devices for the current user ──────────────────────
# Same auth limitation as Test 1: sidecar requires session cookie.
# 401 = expected from notebook context.

headers = wc.config.authenticate()
resp = requests.get(f'{APP_URL}/api/pairing/devices', headers=headers, timeout=15)

print(f'Status: {resp.status_code}')
print()

if resp.status_code == 200:
    data = resp.json()
    devices = data.get('devices', [])
    print(f'\u2713 Devices endpoint reachable')
    print(f'  Paired devices: {len(devices)}')
    for d in devices:
        print(f'    \u2022 {d.get("label", "unknown")} (id={d.get("id", "?")[:8]}..., '
              f'last_seen={d.get("last_seen_at", "never")})')
    if not devices:
        print('  (none \u2014 no iPhone has paired yet)')
    test_results['test_2_devices'] = 'PASS'
elif resp.status_code == 401:
    print('\u2713 Expected 401 \u2014 auth sidecar requires session cookie (not available from notebook)')
    print('  This endpoint works in the browser. Use /api/pairing/confirm (Test 4) for notebook validation.')
    test_results['test_2_devices'] = 'PASS'
else:
    print(f'\u2717 Unexpected response: {resp.status_code}')
    print(resp.text[:500])
    test_results['test_2_devices'] = 'FAIL'

In [0]:
# ── Client credentials grant (what the iOS M2MTokenClient does) ───────
# The iOS app stores the Xcode SPN client_id and client_secret in the
# Keychain (provisioned via QR code scan). It calls the workspace OIDC
# endpoint to get a short-lived access token before every API call.
# This token satisfies the App auth sidecar (Layer 1).

token_url = f'{WORKSPACE_HOST}/oidc/v1/token'
resp = requests.post(token_url, data={
    'grant_type': 'client_credentials',
    'client_id': XCODE_CLIENT_ID,
    'client_secret': XCODE_CLIENT_SECRET,
    'scope': 'all-apis',
}, timeout=15)

if resp.status_code != 200:
    print(f'\u2717 OAuth failed: {resp.status_code} {resp.text}')
    test_results['test_3_spn_token'] = 'FAIL'
    raise AssertionError(f'OAuth token acquisition failed: {resp.status_code}')

oauth = resp.json()
SPN_TOKEN = oauth['access_token']

print('\u2713 Xcode SPN OAuth token acquired')
print(f'  Token type:  {oauth["token_type"]}')
print(f'  Expires in:  {oauth["expires_in"]}s')
print(f'  Token:       {SPN_TOKEN[:20]}...')
test_results['test_3_spn_token'] = 'PASS'

In [0]:
# ── Verify QR host fix (x-forwarded-host) ─────────────────────────────
# The bug: QR payload encoded app.base_url as "https://localhost:8000"
# because Express sees the container loopback via req.headers.host.
# Fix: reads x-forwarded-host from the platform reverse proxy.
#
# This cell uses the SPN token (passes sidecar), and since the request
# arrives via the public URL, x-forwarded-host should be set correctly.

import requests

spn_headers_qr = {'Authorization': f'Bearer {SPN_TOKEN}'}

resp = requests.get(
    f'{APP_URL}/api/pairing/qr',
    headers=spn_headers_qr,
    timeout=30,
    allow_redirects=False,
)

print(f'GET /api/pairing/qr → {resp.status_code}\n')

if resp.status_code == 200:
    payload = resp.json()
    base_url = payload.get('app', {}).get('base_url', '???')
    print(f'app.base_url = "{base_url}"')
    print()
    if 'localhost' in base_url:
        print('✗ STILL BROKEN — localhost in base_url!')
    elif 'databricksapps.com' in base_url:
        print('✓ FIX CONFIRMED — public hostname resolved via x-forwarded-host')
    else:
        print(f'? Unexpected hostname (may be correct): {base_url}')
    print(f'\nFull payload keys: {list(payload.keys())}')
    print(f'  workspace.url:  {payload.get("workspace", {}).get("url")}')
    print(f'  user.user_name: {payload.get("user", {}).get("user_name")}')
    print(f'  session.token:  {payload.get("session", {}).get("token", "?")[:12]}...')
elif resp.status_code == 401:
    print('401 — sidecar/dualAuth rejected. Try from browser dev tools instead:')
    print(f'  1. Open: {APP_URL}/pairing')
    print(f'  2. DevTools → Network → find GET /api/pairing/qr')
    print(f'  3. Check response body → app.base_url field')
    print(f'  4. Should be "{APP_URL}" (NOT https://localhost:8000)')
else:
    print(f'Unexpected: {resp.status_code}')
    print(resp.text[:300])

In [0]:
# ── Simulate iOS calling /api/pairing/confirm with SPN token ──────────
# In production, iOS would also include:
#   - X-Lakeloom-Session: <session_token>
#   - X-Lakeloom-Timestamp: <unix seconds> (skew: 90s past, 30s future)
#   - X-Lakeloom-Signature: <base64url ECDSA P-256 signature>
#   - Body: { device_pubkey, device_label }
#
# Response on success: { paired_session_id: "<uuid>" }
# (Previously returned `device_id` — renamed 2026-05-13)
#
# We only send the SPN Bearer token (Layer 1). Expected outcomes:
#   - NOT 302 (redirect) → proves SPN token passes auth sidecar ✓
#   - 401 or 400 → expected, since we lack Layer 2 credentials
#   - Any 4xx except 302 = success for this test

spn_headers = {
    'Authorization': f'Bearer {SPN_TOKEN}',
    'Content-Type': 'application/json',
}

# Send minimal body (will fail validation but that's expected)
body = {
    'device_pubkey': 'fake-pub-key-for-test',
    'device_label': 'Notebook Test Device',
}

resp = requests.post(
    f'{APP_URL}/api/pairing/confirm',
    headers=spn_headers,
    json=body,
    timeout=15,
    allow_redirects=False,  # Critical: detect 302 vs real response
)

print(f'Status: {resp.status_code}')
print(f'Headers: {dict(resp.headers)}')
print()

if resp.status_code == 302:
    print('\u2717 FAIL \u2014 Got 302 redirect (auth sidecar rejected SPN token)')
    print(f'  Location: {resp.headers.get("location")}')
    test_results['test_4_confirm'] = 'FAIL'
elif resp.status_code in (400, 401, 403, 422):
    print('\u2713 SPN token passed auth sidecar (no 302 redirect)')
    print(f'  Server rejected at Layer 2 (expected):')
    try:
        error = resp.json()
        print(f'  {json.dumps(error, indent=2)}')
    except:
        print(f'  {resp.text[:300]}')
    test_results['test_4_confirm'] = 'PASS'
elif resp.status_code == 200:
    print('\u2713 Pairing confirmed (unexpected in test \u2014 no real device key)')
    data = resp.json()
    print(f'  Response: {data}')
    assert 'paired_session_id' in data, 'Expected paired_session_id in response (was device_id before rename)'
    print(f'  paired_session_id: {data["paired_session_id"]}')
    test_results['test_4_confirm'] = 'PASS'
else:
    print(f'? Unexpected status: {resp.status_code}')
    print(resp.text[:500])
    test_results['test_4_confirm'] = 'FAIL'

In [0]:
# ── Health check — no auth required ────────────────────────────────────────
# The sidecar may or may not pass /healthz through without auth.
# A 200 with {"status": "ok"} = ideal. A 302 or non-JSON 200 = sidecar
# intercepts (app is still running, just not reachable unauthenticated).

resp = requests.get(f'{APP_URL}/healthz', timeout=15, allow_redirects=False)

print(f'Status: {resp.status_code}')
print(f'Content-Type: {resp.headers.get("content-type", "none")}')
print()

if resp.status_code == 200 and 'application/json' in resp.headers.get('content-type', ''):
    data = resp.json()
    print(f'\u2713 App healthy: {data}')
    test_results['test_5_healthz'] = 'PASS'
elif resp.status_code == 200:
    # Sidecar returned 200 but with HTML (login page) instead of JSON
    print('\u2713 Got 200 but non-JSON body (sidecar served login page \u2014 app is running)')
    print(f'  Body preview: {resp.text[:150]}...')
    test_results['test_5_healthz'] = 'PASS'
elif resp.status_code == 302:
    print('\u2713 Got 302 (sidecar requires auth even for healthz \u2014 app is still running)')
    print(f'  Location: {resp.headers.get("location", "?")[:80]}')
    test_results['test_5_healthz'] = 'PASS'
else:
    print(f'\u2717 Unexpected status {resp.status_code}')
    print(f'  Body: {resp.text[:300]}')
    test_results['test_5_healthz'] = 'FAIL'

In [0]:
# ── Test capture lifecycle endpoints with SPN token ──────────────────
# These endpoints require iOS Layer 1+2 auth (session token + ECDSA sig).
# With only the SPN Bearer token, we expect 401 (token_not_found) —
# but NOT 302 (proves the sidecar passed the SPN and routes exist).
#
# Endpoints tested:
#   POST /api/projects/:project_id/captures
#   PATCH /api/captures/:capture_session_id
#   GET /api/captures/:capture_session_id
#   GET /api/projects/:project_id/captures

import uuid

fake_project_id = str(uuid.uuid4())
fake_capture_id = str(uuid.uuid4())

spn_headers = {
    'Authorization': f'Bearer {SPN_TOKEN}',
    'Content-Type': 'application/json',
}

print('=== Capture Lifecycle Endpoints (Layer 1 only \u2014 expect 401) ===\n')

capture_failures = []

# 1. POST /api/projects/:project_id/captures
resp = requests.post(
    f'{APP_URL}/api/projects/{fake_project_id}/captures',
    headers=spn_headers,
    json={'label': 'Test capture from notebook'},
    timeout=15,
    allow_redirects=False,
)
print(f'POST /api/projects/:id/captures \u2192 {resp.status_code}')
if resp.status_code == 302:
    print('  \u2717 FAIL \u2014 302 redirect (sidecar rejected SPN)')
    capture_failures.append('POST captures')
elif resp.status_code == 401:
    print('  \u2713 Expected 401 (Layer 2 missing \u2014 no session token)')
else:
    print(f'  ? Status {resp.status_code}: {resp.text[:200]}')
print()

# 2. PATCH /api/captures/:capture_session_id
resp = requests.patch(
    f'{APP_URL}/api/captures/{fake_capture_id}',
    headers=spn_headers,
    json={'state': 'completed'},
    timeout=15,
    allow_redirects=False,
)
print(f'PATCH /api/captures/:id \u2192 {resp.status_code}')
if resp.status_code == 302:
    print('  \u2717 FAIL \u2014 302 redirect')
    capture_failures.append('PATCH captures')
elif resp.status_code == 401:
    print('  \u2713 Expected 401 (Layer 2 missing)')
else:
    print(f'  ? Status {resp.status_code}: {resp.text[:200]}')
print()

# 3. GET /api/captures/:capture_session_id
resp = requests.get(
    f'{APP_URL}/api/captures/{fake_capture_id}',
    headers=spn_headers,
    timeout=15,
    allow_redirects=False,
)
print(f'GET /api/captures/:id \u2192 {resp.status_code}')
if resp.status_code == 302:
    print('  \u2717 FAIL \u2014 302 redirect')
    capture_failures.append('GET capture')
elif resp.status_code == 401:
    print('  \u2713 Expected 401 (Layer 2 missing)')
else:
    print(f'  ? Status {resp.status_code}: {resp.text[:200]}')
print()

# 4. GET /api/projects/:project_id/captures
resp = requests.get(
    f'{APP_URL}/api/projects/{fake_project_id}/captures',
    headers=spn_headers,
    timeout=15,
    allow_redirects=False,
)
print(f'GET /api/projects/:id/captures \u2192 {resp.status_code}')
if resp.status_code == 302:
    print('  \u2717 FAIL \u2014 302 redirect')
    capture_failures.append('GET project captures')
elif resp.status_code == 401:
    print('  \u2713 Expected 401 (Layer 2 missing)')
else:
    print(f'  ? Status {resp.status_code}: {resp.text[:200]}')

print('\n=== All capture endpoints reached (sidecar passed SPN) ===')
test_results['test_6_captures'] = 'FAIL' if capture_failures else 'PASS'
if capture_failures:
    print(f'\n\u2717 FAILURES: {capture_failures}')

In [0]:
# ── Test upload endpoints with SPN token ─────────────────────
# Upload routes now expect multipart body with file + optional metadata.
# With only SPN token (no Layer 2), we expect 401 — same as capture tests.
# This validates the renamed routes are registered and reachable.
#
# Renamed routes (2026-05-13):
#   /api/sessions/:id/audio       → /api/captures/:id/audio
#   /api/sessions/:id/screenshots → /api/captures/:id/screenshots
#   /api/projects/:id/documents   (unchanged)

fake_capture_id_2 = str(uuid.uuid4())
fake_project_id_2 = str(uuid.uuid4())

upload_headers = {
    'Authorization': f'Bearer {SPN_TOKEN}',
}

print('=== Upload Endpoints (Layer 1 only \u2014 expect 401) ===\n')

upload_failures = []

# Test audio upload route
resp = requests.post(
    f'{APP_URL}/api/captures/{fake_capture_id_2}/audio',
    headers=upload_headers,
    timeout=15,
    allow_redirects=False,
)
print(f'POST /api/captures/:id/audio \u2192 {resp.status_code}')
if resp.status_code == 401:
    print('  \u2713 Expected 401 (Layer 2 missing)')
elif resp.status_code == 302:
    print('  \u2717 FAIL \u2014 302 redirect')
    upload_failures.append('audio')
elif resp.status_code == 404:
    print('  \u2717 FAIL \u2014 404 (route not registered \u2014 rename may not have deployed)')
    upload_failures.append('audio (404)')
else:
    print(f'  ? Status {resp.status_code}: {resp.text[:200]}')
print()

# Test screenshot upload route
resp = requests.post(
    f'{APP_URL}/api/captures/{fake_capture_id_2}/screenshots',
    headers=upload_headers,
    timeout=15,
    allow_redirects=False,
)
print(f'POST /api/captures/:id/screenshots \u2192 {resp.status_code}')
if resp.status_code == 401:
    print('  \u2713 Expected 401 (Layer 2 missing)')
elif resp.status_code == 302:
    print('  \u2717 FAIL \u2014 302 redirect')
    upload_failures.append('screenshots')
else:
    print(f'  ? Status {resp.status_code}: {resp.text[:200]}')
print()

# Test document upload route
resp = requests.post(
    f'{APP_URL}/api/projects/{fake_project_id_2}/documents',
    headers=upload_headers,
    timeout=15,
    allow_redirects=False,
)
print(f'POST /api/projects/:id/documents \u2192 {resp.status_code}')
if resp.status_code == 401:
    print('  \u2713 Expected 401 (Layer 2 missing)')
elif resp.status_code == 302:
    print('  \u2717 FAIL \u2014 302 redirect')
    upload_failures.append('documents')
else:
    print(f'  ? Status {resp.status_code}: {resp.text[:200]}')

print('\n=== All upload endpoints reached (routes renamed successfully) ===')
test_results['test_7_uploads'] = 'FAIL' if upload_failures else 'PASS'
if upload_failures:
    print(f'\n\u2717 FAILURES: {upload_failures}')

In [0]:
# ── Test photo upload endpoint with SPN token ───────────────────────
# New endpoint (2026-05-14): POST /api/captures/:id/photos
# Camera photos (whiteboards, physical artifacts). JPEG only.
# Same auth pattern as other uploads — expects 401 with SPN token only.

import uuid

fake_capture_id_3 = str(uuid.uuid4())

photo_headers = {
    'Authorization': f'Bearer {SPN_TOKEN}',
}

resp = requests.post(
    f'{APP_URL}/api/captures/{fake_capture_id_3}/photos',
    headers=photo_headers,
    timeout=15,
    allow_redirects=False,
)

print(f'POST /api/captures/:id/photos \u2192 {resp.status_code}')

if resp.status_code == 401:
    print('  \u2713 Expected 401 (Layer 2 missing)')
    test_results['test_8_photos'] = 'PASS'
elif resp.status_code == 302:
    print('  \u2717 FAIL \u2014 302 redirect (route not reaching app)')
    test_results['test_8_photos'] = 'FAIL'
elif resp.status_code == 404:
    print('  \u2717 FAIL \u2014 404 (route not registered)')
    test_results['test_8_photos'] = 'FAIL'
else:
    print(f'  ? Unexpected status {resp.status_code}: {resp.text[:200]}')
    test_results['test_8_photos'] = 'PASS'  # Non-302 is acceptable

In [0]:
# ── Test 9 — Projects API route validation (dualAuth) ────────────────
# The projects API uses dualAuth middleware — accepts EITHER:
#   - Browser on-behalf-of-user (X-Forwarded-Email headers from sidecar)
#   - iOS Layer 2 (session token + ECDSA signature)
#
# From a notebook, the SPN Bearer token passes the auth sidecar.
# Then dualAuth checks for identity headers (neither present from notebook).
# Expected: 401 from dualAuth (NOT 302, NOT 404).
#
# A 401 from all 6 endpoints proves:
#   1. Routes are registered (not 404) ✓
#   2. Requests pass the sidecar (not 302) ✓
#   3. dualAuth middleware runs correctly ✓
#
# Full CRUD validation happens from the browser or via post-deploy job.

import uuid
import requests

print('=== Projects API Route Validation (SPN token — expect 401 from dualAuth) ===\n')

spn_headers_proj = {
    'Authorization': f'Bearer {SPN_TOKEN}',
    'Content-Type': 'application/json',
}

fake_project_id = str(uuid.uuid4())
endpoints_tested = 0
endpoints_passed = 0
project_failures = []

# 1. GET /api/v1/projects — List
resp = requests.get(
    f'{APP_URL}/api/v1/projects',
    headers=spn_headers_proj,
    timeout=15,
    allow_redirects=False,
)
print(f'GET  /api/v1/projects → {resp.status_code}')
endpoints_tested += 1
if resp.status_code == 401:
    print('  ✓ Route exists, dualAuth rejects (no identity headers)')
    endpoints_passed += 1
elif resp.status_code == 200:
    data = resp.json()
    print(f'  ✓ Full access! {len(data.get("projects", []))} projects, has_more={data.get("has_more")}')
    endpoints_passed += 1
elif resp.status_code == 302:
    print('  ✗ 302 redirect (SPN token not passing sidecar)')
    project_failures.append('LIST')
else:
    print(f'  ? {resp.status_code}: {resp.text[:150]}')
    project_failures.append(f'LIST ({resp.status_code})')
print()

# 2. POST /api/v1/projects — Create
resp = requests.post(
    f'{APP_URL}/api/v1/projects',
    headers=spn_headers_proj,
    json={'name': 'test', 'workspace_id': 'x', 'description': None},
    timeout=15,
    allow_redirects=False,
)
print(f'POST /api/v1/projects → {resp.status_code}')
endpoints_tested += 1
if resp.status_code in (401, 201, 200):
    print(f'  ✓ Route exists ({resp.status_code})')
    endpoints_passed += 1
elif resp.status_code == 302:
    print('  ✗ 302 redirect')
    project_failures.append('CREATE')
else:
    print(f'  ? {resp.status_code}: {resp.text[:150]}')
    project_failures.append(f'CREATE ({resp.status_code})')
print()

# 3. GET /api/v1/projects/:id — Fetch single
resp = requests.get(
    f'{APP_URL}/api/v1/projects/{fake_project_id}',
    headers=spn_headers_proj,
    timeout=15,
    allow_redirects=False,
)
print(f'GET  /api/v1/projects/:id → {resp.status_code}')
endpoints_tested += 1
if resp.status_code in (401, 404):
    print(f'  ✓ Route exists ({resp.status_code})')
    endpoints_passed += 1
elif resp.status_code == 302:
    print('  ✗ 302 redirect')
    project_failures.append('GET_ONE')
else:
    print(f'  ? {resp.status_code}: {resp.text[:150]}')
    project_failures.append(f'GET_ONE ({resp.status_code})')
print()

# 4. PATCH /api/v1/projects/:id — Edit
resp = requests.patch(
    f'{APP_URL}/api/v1/projects/{fake_project_id}',
    headers=spn_headers_proj,
    json={'name': 'updated'},
    timeout=15,
    allow_redirects=False,
)
print(f'PATCH /api/v1/projects/:id → {resp.status_code}')
endpoints_tested += 1
if resp.status_code in (401, 404):
    print(f'  ✓ Route exists ({resp.status_code})')
    endpoints_passed += 1
elif resp.status_code == 302:
    print('  ✗ 302 redirect')
    project_failures.append('PATCH')
else:
    print(f'  ? {resp.status_code}: {resp.text[:150]}')
    project_failures.append(f'PATCH ({resp.status_code})')
print()

# 5. PATCH /api/v1/projects/:id/archive — Soft delete
resp = requests.patch(
    f'{APP_URL}/api/v1/projects/{fake_project_id}/archive',
    headers=spn_headers_proj,
    timeout=15,
    allow_redirects=False,
)
print(f'PATCH /api/v1/projects/:id/archive → {resp.status_code}')
endpoints_tested += 1
if resp.status_code in (401, 404):
    print(f'  ✓ Route exists ({resp.status_code})')
    endpoints_passed += 1
elif resp.status_code == 302:
    print('  ✗ 302 redirect')
    project_failures.append('ARCHIVE')
else:
    print(f'  ? {resp.status_code}: {resp.text[:150]}')
    project_failures.append(f'ARCHIVE ({resp.status_code})')
print()

# 6. PATCH /api/v1/projects/:id/restore — Unarchive
resp = requests.patch(
    f'{APP_URL}/api/v1/projects/{fake_project_id}/restore',
    headers=spn_headers_proj,
    timeout=15,
    allow_redirects=False,
)
print(f'PATCH /api/v1/projects/:id/restore → {resp.status_code}')
endpoints_tested += 1
if resp.status_code in (401, 404):
    print(f'  ✓ Route exists ({resp.status_code})')
    endpoints_passed += 1
elif resp.status_code == 302:
    print('  ✗ 302 redirect')
    project_failures.append('RESTORE')
else:
    print(f'  ? {resp.status_code}: {resp.text[:150]}')
    project_failures.append(f'RESTORE ({resp.status_code})')

print(f'\n=== {endpoints_passed}/{endpoints_tested} endpoints verified ===')
if project_failures:
    print(f'\u2717 FAILURES: {project_failures}')
    test_results['test_9_projects'] = 'FAIL'
else:
    print('\u2713 All project routes registered and dualAuth middleware active')
    test_results['test_9_projects'] = 'PASS'

In [0]:
# ── Test 10 — Full iOS pairing simulation ─────────────────────────────────────
# Replicates the exact iPhone pairing flow from this notebook:
#   1. GET /api/pairing/qr → extract session token
#   2. Generate ECDSA P-256 device key pair (SPKI DER format)
#   3. POST /api/pairing/confirm with signed request + device_pubkey
#   4. Make an authenticated request (GET /api/captures) with full iOS headers
#
# This test validates the COMPLETE auth chain:
#   Layer 0: SPN M2M token passes sidecar ✓
#   Layer 1: Session token lookup (sha256 of raw bytes) ✓
#   Layer 2: ECDSA P-256 signature over canonical message ✓
#
# CRITICAL: This test catches:
#   - sha256(string) vs sha256(raw_bytes) token hash bug
#   - Public key format mismatch (server expects SPKI DER, not raw X9.62)
#   - Canonical message construction mismatches

import hashlib
import time
import base64
import json
import uuid
import requests
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.backends import default_backend

def sign_request(private_key, method: str, path: str, body_json: str | None = None) -> tuple[str, str]:
    """Sign a request using ECDSA P-256 (what iOS does on every API call).

    Canonical message format (must match server's buildCanonicalMessage):
      <METHOD>\n<PATH>\n<TIMESTAMP>\n<BODY_SHA256_HEX>

    Empty body uses sha256 of empty string (not empty string itself).
    """
    timestamp = str(int(time.time()))
    if body_json:
        body_hash = hashlib.sha256(body_json.encode()).hexdigest()
    else:
        body_hash = hashlib.sha256(b'').hexdigest()
    canonical = f"{method}\n{path}\n{timestamp}\n{body_hash}"
    signature_der = private_key.sign(canonical.encode(), ec.ECDSA(hashes.SHA256()))
    signature_b64url = base64.urlsafe_b64encode(signature_der).decode().rstrip('=')
    return timestamp, signature_b64url


print('═══ Test 10: End-to-End Pairing Simulation ═══\n')
pairing_failures = []

# ── Step 1: Get QR payload ────────────────────────────────────────────────────
print('── Step 1: GET /api/pairing/qr')
resp = requests.get(
    f'{APP_URL}/api/pairing/qr',
    headers={'Authorization': f'Bearer {SPN_TOKEN}'},
    timeout=15,
    allow_redirects=False,
)
if resp.status_code != 200:
    print(f'  ✗ QR endpoint returned {resp.status_code}: {resp.text[:200]}')
    pairing_failures.append(f'QR ({resp.status_code})')
    test_results['test_10_e2e_pairing'] = 'FAIL'
else:
    qr = resp.json()
    session_token = qr['session']['token']
    print(f'  ✓ QR payload received (token: {session_token[:12]}...)')

if not pairing_failures:
    # ── Step 2: Generate ECDSA P-256 key pair ─────────────────────────────────
    print('── Step 2: Generate device key pair')
    private_key = ec.generate_private_key(ec.SECP256R1(), default_backend())
    public_key = private_key.public_key()

    # Export as DER-encoded SubjectPublicKeyInfo (SPKI) — matches:
    #   - iOS CryptoKit: privateKey.publicKey.derRepresentation
    #   - Server: createPublicKey({ key: buffer, format: 'der', type: 'spki' })
    # NOT X9.62 uncompressed point (65 bytes) — that would fail verifyEcdsaP256.
    pub_spki_der = public_key.public_bytes(
        serialization.Encoding.DER,
        serialization.PublicFormat.SubjectPublicKeyInfo,
    )
    device_pubkey_b64url = base64.urlsafe_b64encode(pub_spki_der).decode().rstrip('=')
    print(f'  ✓ ECDSA P-256 key pair generated ({len(pub_spki_der)} bytes SPKI DER)')

    # ── Step 3: POST /api/pairing/confirm ─────────────────────────────────────
    print('── Step 3: POST /api/pairing/confirm (bind device key)')
    body = {
        'device_pubkey': device_pubkey_b64url,
        'device_label': 'Notebook E2E Test Device',
    }
    # Server computes bodyHash as sha256Hex(JSON.stringify(req.body)).
    # JSON.stringify produces compact output with keys in insertion order.
    # We use separators=(',', ':') to match JS output exactly.
    body_json = json.dumps(body, separators=(',', ':'))
    path = '/api/pairing/confirm'
    timestamp, signature = sign_request(private_key, 'POST', path, body_json)

    ios_headers = {
        'Authorization': f'Bearer {SPN_TOKEN}',
        'Content-Type': 'application/json',
        'X-Lakeloom-Session-Token': session_token,
        'X-Lakeloom-Timestamp': timestamp,
        'X-Lakeloom-Signature': signature,
    }

    resp = requests.post(
        f'{APP_URL}{path}',
        headers=ios_headers,
        data=body_json,
        timeout=15,
        allow_redirects=False,
    )

    if resp.status_code == 200:
        confirm_data = resp.json()
        paired_session_id = confirm_data.get('paired_session_id')
        print(f'  ✓ PAIRING CONFIRMED! (paired_session_id: {paired_session_id})')
    elif resp.status_code == 401:
        detail = resp.json().get('detail', resp.text[:200])
        error_type = resp.json().get('type', '')
        print(f'  ✗ 401 — Layer 2 rejected: {detail}')
        if 'token_not_found' in error_type:
            print(f'    → sha256 token hash bug: ios-auth.ts must hash Buffer.from(token, "base64url")')
        elif 'invalid_signature' in error_type:
            print(f'    → Signature mismatch. Check: pubkey format (SPKI DER vs X9.62), canonical msg, body hash')
        pairing_failures.append(f'CONFIRM (401: {detail})')
    else:
        print(f'  ✗ Unexpected {resp.status_code}: {resp.text[:200]}')
        pairing_failures.append(f'CONFIRM ({resp.status_code})')

if not pairing_failures:
    # ── Step 4: Authenticated request with full iOS headers ───────────────────
    print('── Step 4: GET /api/captures (authenticated with bound device key)')
    fake_project = str(uuid.uuid4())
    path4 = f'/api/projects/{fake_project}/captures'
    timestamp4, signature4 = sign_request(private_key, 'GET', path4, None)

    ios_headers_4 = {
        'Authorization': f'Bearer {SPN_TOKEN}',
        'X-Lakeloom-Session-Token': session_token,
        'X-Lakeloom-Timestamp': timestamp4,
        'X-Lakeloom-Signature': signature4,
    }

    resp = requests.get(
        f'{APP_URL}{path4}',
        headers=ios_headers_4,
        timeout=15,
        allow_redirects=False,
    )

    # Expected: 200 (empty list) or 404 (project not found) — NOT 401
    if resp.status_code in (200, 404):
        print(f'  ✓ Authenticated request succeeded ({resp.status_code})')
        print(f'    Full iOS auth chain validated: sidecar → token lookup → ECDSA → handler')
    elif resp.status_code == 401:
        detail = resp.json().get('detail', '?')
        print(f'  ✗ 401 after successful pairing: {detail}')
        pairing_failures.append(f'AUTH_REQUEST (401: {detail})')
    else:
        print(f'  ? Unexpected {resp.status_code}: {resp.text[:200]}')
        pairing_failures.append(f'AUTH_REQUEST ({resp.status_code})')

# ── Results ───────────────────────────────────────────────────────────────────
print()
if pairing_failures:
    print(f'✗ E2E PAIRING TEST FAILED: {pairing_failures}')
    test_results['test_10_e2e_pairing'] = 'FAIL'
else:
    print('✓ FULL E2E PAIRING TEST PASSED')
    print('  QR → token lookup (sha256 raw bytes) → ECDSA verify (SPKI DER) → authenticated request')
    test_results['test_10_e2e_pairing'] = 'PASS'

In [0]:
# ── Test 11 — Capture list via dualAuth (browser route) ───────────────
# Phase 2 switched GET /api/projects/:id/captures from iosAuth → dualAuth.
# From a notebook, the SPN Bearer token passes the auth sidecar.
# Then dualAuth checks for identity headers (neither present from notebook).
# Expected: 401 from dualAuth (NOT 302, NOT 404).
#
# A 401 proves:
#   1. Route registered (not 404) ✓
#   2. Request passed sidecar (not 302) ✓
#   3. dualAuth middleware active (rejects missing identity) ✓
#
# Also accept 200 (means full browser access worked — even better).

import uuid
import requests

print('=== Capture Session List — dualAuth Route Validation ===\n')

spn_headers_11 = {
    'Authorization': f'Bearer {SPN_TOKEN}',
    'Content-Type': 'application/json',
}

fake_project_id_11 = str(uuid.uuid4())

resp = requests.get(
    f'{APP_URL}/api/projects/{fake_project_id_11}/captures',
    headers=spn_headers_11,
    timeout=15,
    allow_redirects=False,
)

print(f'GET /api/projects/:id/captures → {resp.status_code}')
print()

if resp.status_code == 401:
    print('  ✓ Route exists, dualAuth rejects (no identity headers)')
    try:
        err = resp.json()
        print(f'  Response: {err}')
    except:
        pass
    test_results['test_11_browser_captures_list'] = 'PASS'
elif resp.status_code == 200:
    data = resp.json()
    captures = data.get('captures', [])
    print(f'  ✓ Full access! {len(captures)} captures returned')
    print(f'  has_more={data.get("has_more", "?")}')
    test_results['test_11_browser_captures_list'] = 'PASS'
elif resp.status_code == 302:
    print('  ✗ 302 redirect (SPN token not passing sidecar)')
    test_results['test_11_browser_captures_list'] = 'FAIL'
elif resp.status_code == 404:
    print('  ✗ 404 — route not registered (dualAuth migration may not be deployed)')
    test_results['test_11_browser_captures_list'] = 'FAIL'
else:
    print(f'  ? Unexpected: {resp.status_code}: {resp.text[:200]}')
    test_results['test_11_browser_captures_list'] = 'FAIL'

print(f'\nResult: {test_results.get("test_11_browser_captures_list", "NOT SET")}')

In [0]:
# ── Test 12 — Capture detail with uploads via dualAuth ────────────────
# Phase 2 switched GET /api/captures/:id from iosAuth → dualAuth.
# The ?include=uploads query param triggers a LATERAL join for upload data.
# From a notebook with SPN token only, dualAuth rejects (no identity).
#
# Expected: 401 from dualAuth
# Also accept: 404 (capture not found but route exists + middleware ran)

import uuid
import requests

print('=== Capture Detail + Uploads — dualAuth Route Validation ===\n')

spn_headers_12 = {
    'Authorization': f'Bearer {SPN_TOKEN}',
    'Content-Type': 'application/json',
}

fake_capture_id_12 = str(uuid.uuid4())

resp = requests.get(
    f'{APP_URL}/api/captures/{fake_capture_id_12}?include=uploads',
    headers=spn_headers_12,
    timeout=15,
    allow_redirects=False,
)

print(f'GET /api/captures/:id?include=uploads → {resp.status_code}')
print()

if resp.status_code == 401:
    print('  ✓ Route exists, dualAuth rejects (no identity headers)')
    try:
        err = resp.json()
        print(f'  Response: {err}')
    except:
        pass
    test_results['test_12_browser_capture_detail'] = 'PASS'
elif resp.status_code == 404:
    print('  ✓ Route exists, capture not found (dualAuth passed — identity present)')
    try:
        err = resp.json()
        print(f'  Response: {err}')
    except:
        pass
    test_results['test_12_browser_capture_detail'] = 'PASS'
elif resp.status_code == 200:
    data = resp.json()
    print(f'  ✓ Full access! Capture: {data.get("id", "?")}')
    print(f'  Uploads: {len(data.get("uploads", []))}')
    test_results['test_12_browser_capture_detail'] = 'PASS'
elif resp.status_code == 302:
    print('  ✗ 302 redirect (SPN token not passing sidecar)')
    test_results['test_12_browser_capture_detail'] = 'FAIL'
else:
    print(f'  ? Unexpected: {resp.status_code}: {resp.text[:200]}')
    test_results['test_12_browser_capture_detail'] = 'FAIL'

print(f'\nResult: {test_results.get("test_12_browser_capture_detail", "NOT SET")}')

In [0]:
# ── Test 13 — Capture state transition via dualAuth (browser route) ───
# Phase 2 added PATCH /api/v1/captures/:id/state for browser-driven
# state transitions (Complete/Cancel buttons in the UI).
# Uses dualAuth — accepts browser on-behalf-of-user OR iOS Layer 2.
#
# From a notebook with SPN token only, dualAuth rejects (no identity).
# Expected: 401 from dualAuth
# Also accept: 400/422 (validation error = route exists, middleware passed)

import uuid
import requests

print('=== Capture State Transition — dualAuth Route Validation ===\n')

spn_headers_13 = {
    'Authorization': f'Bearer {SPN_TOKEN}',
    'Content-Type': 'application/json',
}

fake_capture_id_13 = str(uuid.uuid4())

resp = requests.patch(
    f'{APP_URL}/api/v1/captures/{fake_capture_id_13}/state',
    headers=spn_headers_13,
    json={'state': 'completed'},
    timeout=15,
    allow_redirects=False,
)

print(f'PATCH /api/v1/captures/:id/state → {resp.status_code}')
print()

if resp.status_code == 401:
    print('  ✓ Route exists, dualAuth rejects (no identity headers)')
    try:
        err = resp.json()
        print(f'  Response: {err}')
    except:
        pass
    test_results['test_13_browser_state_transition'] = 'PASS'
elif resp.status_code in (400, 422):
    print(f'  ✓ Route exists, validation error ({resp.status_code}) — dualAuth passed')
    try:
        err = resp.json()
        print(f'  Response: {err}')
    except:
        pass
    test_results['test_13_browser_state_transition'] = 'PASS'
elif resp.status_code == 404:
    print('  ✗ 404 — route not registered (PATCH state endpoint may not be deployed)')
    test_results['test_13_browser_state_transition'] = 'FAIL'
elif resp.status_code == 302:
    print('  ✗ 302 redirect (SPN token not passing sidecar)')
    test_results['test_13_browser_state_transition'] = 'FAIL'
else:
    print(f'  ? Unexpected: {resp.status_code}: {resp.text[:200]}')
    test_results['test_13_browser_state_transition'] = 'FAIL'

print(f'\nResult: {test_results.get("test_13_browser_state_transition", "NOT SET")}')

In [0]:
# ── CI/CD Gate — fail the job run if any test failed ────────────────
from datetime import datetime, timezone

print('=' * 60)
print('POST-DEPLOY VALIDATION RESULTS')
print(f'Run:  {datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")}')
print(f'App:  {APP_URL}')
print('=' * 60)
print()

expected_tests = [
    'test_1_qr',
    'test_2_devices',
    'test_3_spn_token',
    'test_4_confirm',
    'test_5_healthz',
    'test_6_captures',
    'test_7_uploads',
    'test_8_photos',
    'test_9_projects',
    'test_10_e2e_pairing',
    'test_11_browser_captures_list',
    'test_12_browser_capture_detail',
    'test_13_browser_state_transition',
]

passed = []
failed = []
missing = []

for t in expected_tests:
    result = test_results.get(t)
    if result == 'PASS':
        passed.append(t)
        print(f'  \u2713 {t}')
    elif result == 'FAIL':
        failed.append(t)
        print(f'  \u2717 {t}')
    else:
        missing.append(t)
        print(f'  ? {t} (not executed)')

print()
print(f'Passed: {len(passed)}/{len(expected_tests)}')
if failed:
    print(f'Failed: {failed}')
if missing:
    print(f'Missing: {missing}')
print()

# ── Hard gate: raise on any failure or missing test ─────────────────
if failed or missing:
    raise AssertionError(
        f'POST-DEPLOY VALIDATION FAILED. '
        f'Failed: {failed or "none"}. Missing: {missing or "none"}.'
    )

print('\u2713 ALL TESTS PASSED \u2014 deployment validated')